# Insertion Sort vs. Heap Sort — A CLRS-Faithful Comparison

**Table of Contents**
1. Insertion Sort
2. Heap Sort
3. Comparison Setup
4. Comparison Setup
5. Conclusion

_Most of the code was taken from the pseudocode in CLRS._

## 1. Insertion Sort
Insertion sort is one of the "simpler" type of sorting algorithm. It works by partitioning the array: at any given time the left part of the array is sorted (starting from the key), while the remaining right part is still to be sorted.
*insertion_sort(A)* has a complexity of $\Theta(n^2)$.

In [ ]:
def insertion_sort(A):
    for j in range(1, len(A)):
        key = A[j]
        i = j - 1
        while i > -1 and A[i] > key:
            A[i + 1] = A[i]
            i = i - 1
        A[i + 1] = key
    return A


### Example

In [ ]:
test = [5, 2, 4, 6, 1, 3]
print("before:", test)
insertion_sort(test)
print("after: ", test)

## 2. Heap Sort
Heap sort is a sorting algorithm based on a _max-heap_ data structure. A max-heap is an array with the following properties:
- The root is the maximum element, and is stored at `A[0]`.
- The left child of a node `i` is stored at `2 * i + 1`,
- The right child of a node `i` is stored at `2 * i + 2`,
- The parent of a node `i` is stored at $\lfloor (i - 1) / 2 \rfloor$ (`i//2`).

Heap sort operates in 2 steps:
1. Build a max-heap from the input array.
2. Repeatedly swap the root (the maximum element) with the last element of the active heap, decrease the active heap size by one, and call max_heapify on the new root to restore the max-heap property. Repeat this process until the array is fully sorted.

The complexity of heap sort is $\Theta(n \log n)$, due to the travesal time of the max-heapify being $O(\log n)$ which is done n - 1 times (in short).


In [ ]:
def parent(i):
    return i // 2

def left(i):
    return 2 * i

def right(i):
    return 2 * i + 1


def max_heapify(A, i, heap_size):
    l = left(i)
    r = right(i)

    if l <= heap_size and A[l - 1] > A[i - 1]:
        largest = l
    else:
        largest = i

    if r <= heap_size and A[r - 1] > A[largest - 1]:
        largest = r

    if largest != i:
        A[i - 1], A[largest - 1] = A[largest - 1], A[i - 1]
        max_heapify(A, largest, heap_size)


def build_max_heap(A):
    n = len(A)
    heap_size = n
    for i in range(n // 2, 0, -1):
        max_heapify(A, i, heap_size)
    return heap_size


def heapsort(A):
    heap_size = build_max_heap(A)
    n = len(A)
    for i in range(n, 1, -1):
        A[0], A[i - 1] = A[i - 1], A[0]
        heap_size = heap_size - 1
        max_heapify(A, 1, heap_size)
    return A


### Example

In [ ]:
test = [5, 13, 2, 25, 7, 17, 20, 8, 4]
print("before:", test)
heapsort(test)
print("after: ", test)


## 3. Comparison Setup

To compare the two algorithms, some tweaks are needed. A comparison is counter is added to both algorithms, and the number of comparisons is returned at the end.


In [ ]:
def insertion_sort_counted(A):
    comparisons = 0
    for j in range(1, len(A)):
        key = A[j]
        i = j - 1
        while i > -1:
            comparisons += 1
            if A[i] > key:
                A[i + 1] = A[i]
                i -= 1
            else:
                break
        A[i + 1] = key
    return comparisons


def max_heapify_counted(A, i, heap_size, counter):
    l = left(i)
    r = right(i)

    if l <= heap_size:
        counter[0] += 1
        if A[l - 1] > A[i - 1]:
            largest = l
        else:
            largest = i
    else:
        largest = i

    if r <= heap_size:
        counter[0] += 1
        if A[r - 1] > A[largest - 1]:
            largest = r

    if largest != i:
        A[i - 1], A[largest - 1] = A[largest - 1], A[i - 1]
        max_heapify_counted(A, largest, heap_size, counter)


def heapsort_counted(A):
    counter = [0]
    n = len(A)
    heap_size = n
    for i in range(n // 2, 0, -1):
        max_heapify_counted(A, i, heap_size, counter)
    for i in range(n, 1, -1):
        A[0], A[i - 1] = A[i - 1], A[0]
        heap_size -= 1
        max_heapify_counted(A, 1, heap_size, counter)
    return counter[0]


In [ ]:
import random

random.seed(1)

for n in [10, 100, 1000, 5000]:
    a1 = [random.randint(0, 10000000) for i in range(n)]
    a2 = a1.copy()
    c1 = insertion_sort_counted(a1)
    c2 = heapsort_counted(a2)

    print(f"n = {n} | Insertion: {c1} comparisons | Heap: {c2} comparisons")


It becomes clear that every time n doubles, insertion sort's comparison
count is roughly quadrupled, while heap sort's comparison count is
roughly linearly doubled.
This is expected, as for the theoretic complexity of these two algorithms.


## 4. Empirical Running Time Comparison

Now, to test running times, we measure the time it takes to sort the following types of arrays:
- random arrays,
- already-sorted arrays (insertion sort's best case, `Θ(n)`),
- reverse-sorted arrays (insertion sort's worst case, `Θ(n²)`),


In [ ]:
import time
import copy

def make_arrays(n, kind):
    if kind == "random":
        return [random.randint(0, 10000000) for i in range(n)]
    elif kind == "sorted":
        return list(range(n))
    else:
        return list(range(n, 0, -1))

def time_sort(sort_fn, A):
    A = A.copy()
    start = time.perf_counter()
    sort_fn(A)
    end = time.perf_counter()
    return end - start, A


In [ ]:
random.seed(1)

sizes = [10, 100, 500, 1000, 2000, 3000, 4000, 5000]
kinds = ["random", "sorted", "reverse"]

results = {kind: {"insertion": [], "heap": []} for kind in kinds}

for kind in kinds:
    for n in sizes:
        base = make_arrays(n, kind)

        t_ins, sorted_ins = time_sort(insertion_sort, base)
        t_heap, sorted_heap = time_sort(heapsort, base)

        results[kind]["insertion"].append(t_ins)
        results[kind]["heap"].append(t_heap)

    print(f"--- {kind} ---")
    for n, t_i, t_h in zip(sizes, results[kind]["insertion"], results[kind]["heap"]):
        print(f"n={n:>5}  insertion={t_i:.6f}s   heap={t_h:.6f}s")
    print()


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

titles = {
    "random": "Random arrays",
    "sorted": "Already-sorted arrays (insertion best case)",
    "reverse": "Reverse-sorted arrays (insertion worst case)",
}

for ax, kind in zip(axes, kinds):
    ax.plot(sizes, results[kind]["insertion"], marker="o", label="Insertion sort")
    ax.plot(sizes, results[kind]["heap"], marker="o", label="Heap sort")
    ax.set_title(titles[kind])
    ax.set_xlabel("n (array size)")
    ax.set_ylabel("time (seconds)")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 5. Conclusion

**Insertion sort**: In the best case the complexity is Θ(n), while it is Θ(n²) worst/average case.
On an already-sorted array the algorithm only does one comparison per element
Θ(n) total. On a reverse-sorted array, every new key is smaller than everything
already placed, so it must be shifted all the way to the front of the sorted
part; the number of comparisons grows like $1+2+\ldots+ (n-1) = \Theta(n^2)$.

**Heap sort — Θ(n log n)**:
Heap sort's running time doesn't really depend on the input order. Building the
max-heap is $\Theta(n)$, and then each of the n-1 max extractions costs $O(\log n)$, giving Θ(n log n) regardless of whether the input was
random, sorted, or reverse-sorted.

So the right choice depends on context: heap sort for guaranteed Θ(n log n)
performance regardless of input, insertion sort for small or nearly-sorted
inputs where its low overhead and Θ(n) best case shine.

So, asymptotically speaking, heap sort is strictly better, as shown in the graphs above, but there are some cases where insertion sort can still be vaild: on an already sorted (or nearly), it is better with just $O(n)$. The right choice depends on context: heap sort for guaranteed Θ(n log n)
performance regardless of input, insertion sort for small or nearly-sorted
inputs where its low overhead and Θ(n) can be put to use.